# Studi Kasus Webinar ke-6: Klasifikasi Dampak Bencana Berbasis Citra Satelit dan AI
## CNN untuk Mengenali Kondisi Wilayah Setelah Bencana

Notebook ini mengadaptasi tema **AI dan analisis citra satelit** menjadi studi kasus klasifikasi kondisi wilayah untuk mendukung penilaian awal pascabencana. CNN digunakan untuk membedakan lima pola ilustratif: **banjir, longsor, area terbakar, vegetasi sehat, dan kawasan terbangun**.

**Penting:** Agar dapat dijalankan tanpa mengunduh dataset eksternal, notebook menghasilkan citra sintetis dengan pola sederhana. Data ini bukan citra satelit sungguhan dan hasilnya tidak boleh digunakan untuk keputusan tanggap darurat nyata. Implementasi lapangan memerlukan citra satelit autentik, label terverifikasi, serta validasi oleh petugas/ahli.

## 1. Instalasi pustaka
Jalankan jika pustaka belum tersedia.

In [ ]:
%pip install -q tensorflow scikit-learn matplotlib

## 2. Import pustaka

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import pandas as pd

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = [
    "Banjir", "Longsor", "Area Terbakar",
    "Vegetasi Sehat", "Kawasan Terbangun"
]
IMG_SIZE = 64
print("TensorFlow:", tf.__version__)
print("Kelas kondisi wilayah:", CLASS_NAMES)

## 3. Studi kasus dan pembuatan data sintetis

Skenario: tim pemantauan ingin membuat prototipe yang mengelompokkan potongan citra wilayah ke dalam lima kategori kondisi. Data sintetis berikut hanya untuk mempraktikkan alur machine learning, bukan merepresentasikan tekstur dan spektrum citra satelit sebenarnya.

In [ ]:
def make_patch(label, seed, size=64):
    """Membuat patch RGB sintetis untuk demonstrasi klasifikasi kondisi wilayah."""
    rng = np.random.default_rng(seed)
    h = w = size
    img = np.zeros((h, w, 3), dtype=np.float32)
    # Latar vegetasi dengan variasi warna ringan
    img[:] = [rng.uniform(.12, .24), rng.uniform(.30, .48), rng.uniform(.10, .22)]
    yy, xx = np.mgrid[0:h, 0:w]

    if label == 0:  # banjir: area air berwarna biru gelap
        cx, cy = int(rng.integers(20, 44)), int(rng.integers(20, 44))
        rx, ry = int(rng.integers(15, 27)), int(rng.integers(12, 24))
        mask = ((xx-cx)/rx)**2 + ((yy-cy)/ry)**2 < 1
        img[mask] = [.10, .28, .48]
        # garis tepi air yang lebih terang
        edge = (((xx-cx)/rx)**2 + ((yy-cy)/ry)**2 < 1.18) & (~mask)
        img[edge] = [.38, .43, .35]
    elif label == 1:  # longsor: bekas tanah terbuka memanjang di lereng
        for _ in range(2):
            x0, y0 = int(rng.integers(8, 30)), int(rng.integers(0, 20))
            for ypix in range(y0, min(h, y0+48)):
                xpix = x0 + int((ypix-y0)*rng.uniform(.25,.55))
                width = int(rng.integers(3, 8))
                img[ypix, max(0,xpix-width):min(w,xpix+width), :] = [.58,.39,.24]
    elif label == 2:  # area terbakar: bercak gelap dengan tepi cokelat
        cx, cy = int(rng.integers(18, 46)), int(rng.integers(18, 46))
        rx, ry = int(rng.integers(12, 23)), int(rng.integers(10, 20))
        dist = ((xx-cx)/rx)**2 + ((yy-cy)/ry)**2 + rng.normal(0,.12,(h,w))
        img[dist < .72] = [.10,.10,.09]
        img[(dist >= .72) & (dist < 1.15)] = [.48,.25,.12]
    elif label == 3:  # vegetasi sehat: tekstur hijau berbintik
        img[:] = [.10,.42,.16]
        for _ in range(90):
            cx, cy = int(rng.integers(0,w)), int(rng.integers(0,h))
            r = int(rng.integers(1,4))
            mask = (xx-cx)**2 + (yy-cy)**2 <= r*r
            img[mask] = [rng.uniform(.08,.20), rng.uniform(.38,.62), rng.uniform(.08,.22)]
    else:  # kawasan terbangun: blok dan garis jalan
        img[:] = [.48,.45,.40]
        for _ in range(12):
            x0, y0 = int(rng.integers(0,54)), int(rng.integers(0,54))
            rw, rh = int(rng.integers(4,11)), int(rng.integers(4,10))
            img[y0:min(h,y0+rh), x0:min(w,x0+rw)] = [.72,.67,.57]
        img[:, 29:34, :] = [.25,.25,.24]
        img[30:34, :, :] = [.25,.25,.24]

    img += rng.normal(0,.035,img.shape)
    return np.clip(img,0,1).astype(np.float32)

samples_per_class = 240
X, y = [], []
for c in range(len(CLASS_NAMES)):
    for i in range(samples_per_class):
        X.append(make_patch(c, SEED + c*10000 + i))
        y.append(c)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)
idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]
print("Bentuk citra:", X.shape, "| Bentuk label:", y.shape)
print("Jumlah sampel per kelas:", samples_per_class)

## 4. Contoh citra sintetis dari setiap kelas

Periksa pola buatan untuk tiap label sebelum model dilatih. Pada data nyata, tampilan dapat berubah akibat sensor, resolusi, tutupan awan, musim, dan kondisi geografis.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for c, ax in enumerate(axes):
    ix = np.where(y == c)[0][0]
    ax.imshow(X[ix])
    ax.set_title(CLASS_NAMES[c])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Membagi data

Data dibagi menjadi train, validation, dan test dengan stratifikasi kelas. Untuk evaluasi nyata, pembagian sebaiknya juga mempertimbangkan wilayah dan waktu pengambilan citra agar data yang berdekatan tidak bocor ke beberapa subset.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=.50, random_state=SEED, stratify=y_temp
)
print("Train:", len(X_train), "| Validation:", len(X_val), "| Test:", len(X_test))

## 6. Membangun model CNN

CNN mengekstraksi pola spasial dari citra RGB berukuran 64 × 64, lalu menghasilkan probabilitas untuk lima kelas kondisi wilayah.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(.05),
    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(.30),
    layers.Dense(64, activation="relu"),
    layers.Dense(len(CLASS_NAMES), activation="softmax")
])
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

## 7. Melatih model

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=12, batch_size=32,
    callbacks=[early_stop], verbose=1
)

## 8. Grafik proses pelatihan

In [ ]:
hist = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(hist["accuracy"], label="Train")
axes[0].plot(hist["val_accuracy"], label="Validation")
axes[0].set_title("Akurasi"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(hist["loss"], label="Train")
axes[1].plot(hist["val_loss"], label="Validation")
axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout()
plt.show()

## 9. Evaluasi pada data test

Gunakan precision, recall, F1-score, dan confusion matrix per kelas. Skor pada citra sintetis hanya menunjukkan apakah model mempelajari pola buatan; skor tersebut bukan bukti kinerja pada citra satelit nyata.

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
probs = model.predict(X_test, verbose=0)
y_pred = probs.argmax(axis=1)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print(classification_report(
    y_test, y_pred, target_names=CLASS_NAMES, zero_division=0
))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(9,7))
disp.plot(ax=ax, xticks_rotation=35, colorbar=False)
plt.title("Confusion Matrix — Data Test (Sintetis)")
plt.tight_layout()
plt.show()

## 10. Prediksi satu citra sintetis

Bagian ini menampilkan satu contoh dari data test dan membandingkan label sintetis dengan prediksi model.

In [ ]:
sample_index = 0
image = X_test[sample_index]
true_id = int(y_test[sample_index])
p = model.predict(image[None, ...], verbose=0)[0]
pred_id = int(np.argmax(p))

plt.figure(figsize=(4,4))
plt.imshow(image)
plt.axis("off")
plt.title("Label asli: " + CLASS_NAMES[true_id])
plt.show()

print("Label asli:", CLASS_NAMES[true_id])
print("Prediksi:", CLASS_NAMES[pred_id])
print(f"Confidence: {p[pred_id]:.2%}")
display(pd.DataFrame({
    "Kelas": CLASS_NAMES,
    "Probabilitas": p
}).sort_values("Probabilitas", ascending=False).style.format({"Probabilitas":"{:.2%}"}))

## 11. Fungsi prediksi dan menyimpan model

Fungsi menerima citra RGB berukuran 64 × 64 dalam rentang 0–1 atau 0–255. Fungsi ini ditujukan untuk demonstrasi dengan pola sintetis, bukan untuk penilaian dampak bencana di lapangan.

In [ ]:
def predict_condition(image_array):
    image_array = np.asarray(image_array, dtype=np.float32)
    if image_array.shape != (IMG_SIZE, IMG_SIZE, 3):
        raise ValueError(f"Citra harus berbentuk {(IMG_SIZE, IMG_SIZE, 3)}.")
    if image_array.max() > 1:
        image_array = image_array / 255.0
    prob = model.predict(image_array[None, ...], verbose=0)[0]
    best = int(prob.argmax())
    return {
        "prediksi_kondisi_demo": CLASS_NAMES[best],
        "confidence": float(prob[best]),
        "peringatan": "Hasil model sintetis, bukan penilaian dampak bencana nyata."
    }

print(predict_condition(X_test[0]))
model.save("cnn_disaster_condition_synthetic_demo.keras")
print("Model tersimpan: cnn_disaster_condition_synthetic_demo.keras")

## 12. Analisis dan keterbatasan

### Prinsip yang dipelajari
- CNN mempelajari fitur spasial pada citra.
- Dataset harus memiliki label yang konsisten dan representatif.
- Data dibagi menjadi train, validation, dan test.
- Accuracy, precision, recall, F1-score, dan confusion matrix memberikan gambaran evaluasi yang lebih lengkap.

### Keterbatasan
1. Dataset dibuat secara sintetis, bukan citra Sentinel atau citra udara autentik.
2. Pola visual sengaja disederhanakan, sehingga skor tinggi tidak menunjukkan kemampuan mengenali kondisi bencana nyata.
3. Banjir, tanah terbuka, bekas kebakaran, dan kawasan terbangun dapat memiliki tampilan yang tumpang tindih.
4. Citra optik dapat terganggu awan dan bayangan; kombinasi data optik, radar, dan informasi lapangan dapat dibutuhkan.
5. Penggunaan nyata memerlukan data representatif, pengujian lintas wilayah/waktu, dan verifikasi oleh ahli.

### Pengembangan selanjutnya
- Mengganti citra sintetis dengan dataset citra satelit yang memiliki lisensi dan label jelas.
- Melakukan evaluasi per wilayah dan periode waktu.
- Menampilkan prediksi sebagai dukungan keputusan awal, bukan pengganti pemeriksaan lapangan.